# 01 Generate data (the simulator)

**Task.** Build random farming communities, simulate them day by day, and split what comes out into:
* `data/measurements/`: only what a field team measures;
* `data/sketch/`: the channel sketch, zero patterns only;
* `data/truth/`: the networks and the full matrices, read later **only** by notebook 04 for scoring.

Nothing in this notebook uses the test. Model (1)-(2) of the paper is implemented in `lib/community.py`:
$$x[k+1]=\mathbf\Phi x[k]+\mathbf\Gamma u[k]+\mathbf F\omega[k]+w[k],\qquad y[k]=\mathbf Cx[k]+v[k],$$
$$(\mathbf\Phi)_{ii}=\mathbf\Phi_i,\qquad(\mathbf\Phi)_{ij}=W_{ij}\mathbf H_i+P_{ij}\mathbf E_i\quad(i\neq j).$$


In [1]:
import sys, os, glob, numpy as np
sys.path.insert(0, 'lib')
import community as C, farmlib as FL
np.set_printoptions(precision=2, suppress=True, linewidth=150)

CFG = dict(N=8, communities=list(range(10)), years=5, scenario='regional', seepage=0.0)
# Local-aquifer runs (physical network P): scenario='local', seepage=0.0 or 0.5.
for d in ('data/measurements', 'data/truth', 'data/sketch', 'results'):
    os.makedirs(d, exist_ok=True)
CFG

{'N': 8,
 'communities': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'years': 5,
 'scenario': 'regional',
 'seepage': 0.0}

## The farms and their channels
Each farm has 8 channels (Appendix C): soil moisture `sm`, local groundwater level `gw`, perceived scarcity `bel`,
pumping `pump`, spraying `spr`, phone talk `talk` (**never measured**), canopy temperature `temp` and price attention
`pr` (two decoys). The excitations are $d=0$, the canal-turn deviation $e_i^{\rm canal}$ (on the farm's weekly turn
day), and $d=1$, the pest-pressure innovation $e_i^{\rm pest}$ (daily).

In [2]:
# [Model (1)-(2)] build one community: random social network W, heterogeneous farms
Cm = C.build(CFG['N'], seed=0, scenario=CFG['scenario'], seepage=CFG['seepage'])
print('W (truth). Row i = whom farm i follows; rows sum to 1:')
print(Cm['W'])
print('row sums:', Cm['W'].sum(1))

W (truth). Row i = whom farm i follows; rows sum to 1:
[[0.   0.43 0.   0.   0.35 0.   0.   0.22]
 [0.66 0.   0.34 0.   0.   0.   0.   0.  ]
 [0.   0.63 0.   0.   0.   0.   0.   0.37]
 [0.38 0.38 0.24 0.   0.   0.   0.   0.  ]
 [0.   0.   0.23 0.34 0.   0.   0.43 0.  ]
 [0.   0.   0.77 0.   0.   0.   0.23 0.  ]
 [0.   0.   0.   0.   0.   0.77 0.   0.23]
 [0.42 0.   0.3  0.   0.   0.   0.29 0.  ]]
row sums: [1. 1. 1. 1. 1. 1. 1. 1.]


In [3]:
# [A1, common structure] farms have different values but the same zeros
Phi_1, H_1, Gam_1, F_1, p_1 = Cm['agents'][0]
Phi_2, H_2, Gam_2, F_2, p_2 = Cm['agents'][1]
print('same zeros in Phi_1 and Phi_2 :', np.array_equal(Phi_1 != 0, Phi_2 != 0))
print('same values in Phi_1 and Phi_2:', np.allclose(Phi_1, Phi_2))
print('delta_b (pumping from perceived scarcity), farms 1 and 2:', round(p_1['delta_b'], 3), round(p_2['delta_b'], 3))

same zeros in Phi_1 and Phi_2 : True
same values in Phi_1 and Phi_2: False
delta_b (pumping from perceived scarcity), farms 1 and 2: 0.205 0.201


In [4]:
# [stability, truth side] the guarantee of Thm 2 does not need stability; the power prediction (7) does
rho = max(abs(np.linalg.eigvals(Cm['Phi'])))
print('spectral radius of the true Phi:', round(rho, 3))

spectral radius of the true Phi: 0.966


## Simulate, then split
`community.simulate` returns everything, including the true innovations (`D['truth']`). `FL.save_measurements`
writes only the measurable parts and drops `D['truth']`. The networks and matrices go to a separate file.

In [5]:
# [Model (1)] simulate one community and look at what comes out
D = C.simulate(Cm, CFG['years'] * 365, seed=1000)
print('keys returned by the simulator:', list(D.keys()))
print('measurable channels Y         :', list(D['Y'].keys()), '(no talk: phone talk is never measured)')
print('excitations M_d               :', {d: FL.EXCITATIONS[d] for d in D['M']})
print('shared series Omega           :', list(D['Omega'].keys()))
print('event days per farm per year  : canal', D['Pi'][0][:365].sum(0)[0], '| pest', D['Pi'][1][:365].sum(0)[0])

keys returned by the simulator: ['Y', 'avail', 'M', 'Pi', 'sched', 'Omega', 'truth']
measurable channels Y         : ['sm', 'gw', 'bel', 'pump', 'spr', 'temp', 'pr'] (no talk: phone talk is never measured)
excitations M_d               : {0: 'canal-turn deviation', 1: 'pest-pressure innovation'}
shared series Omega           : ['rain', 'et', 'temp', 'pr', 'Q', 'gw', 'pest']
event days per farm per year  : canal 52 | pest 365


In [6]:
# write all communities: measurements / truth, plus the channel sketch
for s in CFG['communities']:
    Cm = C.build(CFG['N'], seed=s, scenario=CFG['scenario'], seepage=CFG['seepage'])
    D = C.simulate(Cm, CFG['years'] * 365, seed=1000 + s)
    FL.save_measurements(f'data/measurements/community_{s:02d}.npz', D)   # D['truth'] is NOT written
    FL.save_truth(f'data/truth/community_{s:02d}.npz', Cm)
FL.save_sketch('data/sketch/channel_sketch.npz')       # zero patterns of Phi_i, H_i, Gamma_i, E_i only
print(len(glob.glob('data/measurements/*.npz')), 'measurement files,', len(glob.glob('data/truth/*.npz')), 'truth files')

10 measurement files, 10 truth files


In [7]:
# check the contract: a measurement file contains no network and no parameters
z = np.load('data/measurements/community_00.npz')
print(sorted(z.keys()))
assert not any(k in z for k in ('W', 'P', 'Phi', 'Gamma')), 'truth leaked into the measurements'
sk = FL.load_sketch('data/sketch/channel_sketch.npz')
print('sketch arrays:', {k: v.dtype for k, v in sk.items()}, '(booleans: zeros only)')

['M_0', 'M_1', 'Omega_Q', 'Omega_et', 'Omega_gw', 'Omega_pest', 'Omega_pr', 'Omega_rain', 'Omega_temp', 'Pi_0', 'Pi_1', 'Y_bel', 'Y_gw', 'Y_pr', 'Y_pump', 'Y_sm', 'Y_spr', 'Y_temp', 'avail_bel', 'avail_gw', 'avail_pr', 'avail_pump', 'avail_sm', 'avail_spr', 'avail_temp', 'sched']
sketch arrays: {'Phi': dtype('bool'), 'H': dtype('bool'), 'Gamma': dtype('bool'), 'E': dtype('bool')} (booleans: zeros only)
